# Conversation State and Chat Interfaces

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

client = OpenAI()
model = "gpt-6-luna"

## LLM Calls are stateless
An LLM does not automatically remember previous API requests. Let's prove it by sending two separate requests.

## First request

First, we tell the LLM our name.

In [ ]:
first_response = client.responses.create(
    model=model,
    input="My name is Lukas.",
)

print(first_response.output_text)

## Second request

Now we ask for the name in a new request. We send only the new message, so no conversation history and no reference to the first response.

In [ ]:
second_response = client.responses.create(
    model=model,
    input="What's my name?",
)

print(second_response.output_text)

## What happened?

Each LLM request is independent and stateless. The second request contains only `What's my name?`, so the model does not know that we said `My name is Lukas.` in the first request.

Reusing the same `client` does not create memory. To build a conversation, our application must explicitly provide the earlier context or connect the responses.

## Add conversation history manually

Our application can create a conversation by storing each turn in a list. Before sending the next request, we add the first user message, all items from the model's response, and the new user message to that list.

In [ ]:
conversation_history = [
    {"role": "user", "content": "My name is Lukas."},
    {"role": "assistant", "content": "Nice to meet you, Lukas!"},
    {"role": "user", "content": "What's my name?"},
]

response_with_history = client.responses.create(
    model=model,
    input=conversation_history,
)

print(response_with_history.output_text)

## What changed?

The model can now answer the question because `conversation_history` contains the earlier context. The model still did not remember anything by itself—our application stored the conversation and sent it again with the new message.

# Other approaches

OpenAI also offers two ways to manage this state for us: `previous_response_id` links one response to the next, while the Conversations API stores a durable conversation that can be reused across sessions.

The tradeoff is vendor lock-in: both approaches rely on OpenAI-specific APIs. If we switch to the Anthropic API, we must replace this state-management code. Managing the history ourselves is more portable, although each provider still uses a slightly different message format.

In [ ]:
# Let OpenAI link this request to the first response
response_with_previous_id = client.responses.create(
    model=model,
    previous_response_id=first_response.id,
    input="What's my name?",
)

print(response_with_previous_id.output_text)

In [ ]:
# Let OpenAI store the conversation under one durable ID
conversation = client.conversations.create()

client.responses.create(
    model=model,
    conversation=conversation.id,
    input="My name is Lukas.",
)

response_with_conversation = client.responses.create(
    model=model,
    conversation=conversation.id,
    input="What's my name?",
)

print(response_with_conversation.output_text)

## Build a user interface with Gradio

Gradio is a Python library for quickly turning a Python function into a web interface. `gr.ChatInterface` creates a chat UI and calls our function with the user's latest message and the conversation history.

In [ ]:
import gradio as gr


def chat(message, history):
    # Gradio provides the history, but this first version deliberately ignores it
    response = client.responses.create(
        model=model,
        input=message,
    )
    return response.output_text


demo = gr.ChatInterface(
    fn=chat,
    title="Stateless AI Chatbot",
    description="The interface shows previous messages, but they are not sent to the model yet.",
)
demo.launch()

In [ ]:
def chat_with_history(message, history):
    # Convert Gradio's UI messages into the simpler format expected by OpenAI
    conversation = [
        {
            "role": history_item["role"],
            "content": history_item["content"][0]["text"],
        }
        for history_item in history
    ]
    conversation.append(
        {"role": "user", "content": message},
    )
    response = client.responses.create(
        model=model,
        input=conversation,
    )
    return response.output_text


demo_with_history = gr.ChatInterface(
    fn=chat_with_history,
    title="AI Chatbot with Conversation History",
)
demo_with_history.launch()

In [ ]:
def chat_with_streaming(message, history):
    conversation = [
        {
            "role": history_item["role"],
            "content": history_item["content"][0]["text"],
        }
        for history_item in history
    ]
    conversation.append(
        {"role": "user", "content": message},
    )

    stream = client.responses.create(
        model=model,
        input=conversation,
        stream=True,
    )

    response_text = ""
    for event in stream:
        if event.type == "response.output_text.delta":
            response_text += event.delta
            yield response_text


demo_with_streaming = gr.ChatInterface(
    fn=chat_with_streaming,
    title="Streaming AI Chatbot",
)
demo_with_streaming.launch()